<p>
  <b>AI Lab: Deep Learning for Computer Vision</b><br>
  <b><a href="https://www.wqu.edu/">WorldQuant University</a></b>
</p>

<div class="alert alert-success" role="alert">
  <p>
    <center><b>Usage Guidelines</b></center>
  </p>
  <p>
    This file is licensed under <a href="https://creativecommons.org/licenses/by-nc-nd/4.0/">Creative Commons Attribution-NonCommercial-NoDerivatives 4.0 International</a>.
  </p>
  <p>
    You <b>can</b>:
    <ul>
      <li><span style="color: green">✓</span> Download this file</li>
      <li><span style="color: green">✓</span> Post this file in public repositories</li>
    </ul>
    You <b>must always</b>:
    <ul>
      <li><span style="color: green">✓</span> Give credit to <a href="https://www.wqu.edu/">WorldQuant University</a> for the creation of this file</li>
      <li><span style="color: green">✓</span> Provide a <a href="https://creativecommons.org/licenses/by-nc-nd/4.0/">link to the license</a></li>
    </ul>
    You <b>cannot</b>:
    <ul>
      <li><span style="color: red">✗</span> Create derivatives or adaptations of this file</li>
      <li><span style="color: red">✗</span> Use this file for commercial purposes</li>
    </ul>
  </p>
  <p>
    Failure to follow these guidelines is a violation of your terms of service and could lead to your expulsion from WorldQuant University and the revocation your certificate.
  </p>
</div>

# 3.2. Traffic Data as Images and Video

<hr>

**Summary:** In this lesson, we'll learn how to work with image and video data for object detection. We'll explore the specific traffic dataset for this project. The focus will be on how video data can be converted to images, understanding bounding boxes for object classification, and using XML annotations to represent those bounding boxes.

**Objectives:**

- Load and organize the project dataset, separating images and their corresponding XML annotations.
- Extract frames from a video at regular intervals.
- Parse XML annotations. 
- Visualize bounding boxes on image data.

**New Terms:**

- Bounding boxes
- Frame rate
- XML
 
<hr>

# Getting Ready

Before we can start this lesson, we need to import the required libraries.

In [ ]:
import sys
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pytubefix
import torch
import torchvision
from IPython.display import Video, VimeoVideo
from pytubefix import YouTube
from torchvision import transforms
from torchvision.io import read_image
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import draw_bounding_boxes, make_grid

Next, print the version numbers of the primary software to improve reproducibility. 

In [ ]:
print("Platform:", sys.platform)
print("Python version:", sys.version)
print("---")
print("CV2 version : ", cv2.__version__)
print("pytubefix version : ", pytubefix.__version__)
print("torch version : ", torch.__version__)
print("torchvision version : ", torchvision.__version__)

It's a good idea to delete any files that were created from a previous run. The following lines do that.

In [ ]:
!rm -rf data_images
!rm -rf data_video

 We are ready to start looking at the data. 🏎️💨

# Exploring Our Data

In this project, we'll work with two datasets. Let's start with the [Dhaka AI dataset](https://www.kaggle.com/datasets/rifat963/dhakaai-dhaka-based-traffic-detection-dataset), which contains images of vehicles in urban traffic scenes from Dhaka, Bangladesh. This dataset is particularly interesting for computer vision as it captures the unique characteristics of Dhaka's busy streets, including a diverse mix of vehicle types and dense traffic conditions.

We'll use the dataset for object detection which is a more complex task than image classification. Object detection identifies specific objects within an image (e.g., cars, buses, motorcycles), determines the precise location of these objects, and draws a bounding box around each detected object.

This dataset will be used in a later lesson to create a custom model. For now, we'll begin by downloading and exploring the dataset. The images and their corresponding annotations are stored in a compressed tarball file, which we'll download and extract to start our analysis.

In [ ]:
!gcloud storage cp gs://wqu-cv-course-datasets/project3.tar.gz . --no-clobber

After downloading the tarball, we can expand it to get our data.

In [ ]:
!tar --skip-old-files -xzf project3.tar.gz

We now longer need the tarball so it can be deleted.

In [ ]:
!rm project3.tar.gz

In [ ]:
VimeoVideo("1009518443", h="607de7238f", width=600)

**Task 3.2.1:** Create a variable for the train directory using `pathlib` syntax.

In [ ]:
dhaka_image_dir = Path("data_images", "train")  # REMOVERHS

print("Data directory:", dhaka_image_dir)

Let's examine some of the contents of the train directory. You'll see two types of files:

1. `.xml` files: These contain the annotations for the images.
2. `.jpg` files: These are the actual image files.

Each image typically has a corresponding XML file.

In [ ]:
dhaka_files = list(dhaka_image_dir.iterdir())
dhaka_files[-5:]

Even though we only see one type of image file, it turns out that the image files can have many different possible extensions. Let's count the file extensions by type and print the results.

In [ ]:
file_extension_counts = Counter(Path(file).suffix for file in dhaka_files)

for extension, count in file_extension_counts.items():
    print(f"Files with extension {extension}: {count}")

# Separating images and bounding boxes data

Bounding boxes are rectangles around a detected object. All bounding box information is contained in the `.xml` files. The images have several different extensions.   It makes sense to separate the different file types into different folders. We'll want to put all `.xml` files in an annotations folder and the various image types in an images folder.

In [ ]:
VimeoVideo("1009519346", h="900f85d512", width=600)

**Task 3.2.2:** Create variables for the images and annotations directories using `pathlib` syntax.

In [ ]:
images_dir = dhaka_image_dir / "images"  # REMOVERHS
annotations_dir = dhaka_image_dir / "annotations"  # REMOVERHS

images_dir.mkdir(exist_ok=True)
annotations_dir.mkdir(exist_ok=True)

In [ ]:
VimeoVideo("1009519352", h="ea89805c63", width=600)

**Task 3.2.3:** Move files to the appropriate directory based on file extensions.

In [ ]:
for file in dhaka_files:
    if file.suffix.lower() in (".jpg", ".jpeg", ".png"):
        target_dir = images_dir  # REMOVERHS
    elif file.suffix.lower() == ".xml":
        target_dir = annotations_dir  # REMOVERHS
    file.rename(target_dir / file.name)

Let's confirm that all the files where moved by making sure there is equal number of images and annotations. 

In [ ]:
images_files = list(images_dir.iterdir())
annotations_files = list(annotations_dir.iterdir())

assert len(images_files) == len(annotations_files)

# Annotations

The annotations are the labels for the data. Each image has an annotation that contains the coordinates and type of object for each bounding box in a given image.

Let's look at the structure of the annotations by loading the first 25 lines of a file. The annotations are stored as XML which is a way to store structured documents. The `<annotation>` tag is the root element, containing all the information about this particular image annotation. The tags within store other information such as `<folder>`. The most important tag for the current project is the `<object>`. It describes an object detected in the image, this associated image contains a "bus". The tag `<bndbox>` is the bounding box information. There are the coordinates of a rectangle surrounding the bus in the image (in pixels): `<xmin>` is the left edge, `<ymin>` is the top edge, `<xmax>` is the right edge, and `<ymax>` is the bottom edge.

In [ ]:
xml_filepath = annotations_dir / "01.xml"
!head -n 25 $xml_filepath

The ElementTree (ET) module in Python can parse an XML file. In XML, the root is the top-level element that contains all other elements. The `tag` attribute contains the name of the element. 

In [ ]:
tree = ET.parse(xml_filepath)
root = tree.getroot()
print(root.tag)

The `find` method is used to locate the first occurrence of a sub-element with a given tag. Let's find the width and height of the image.

In [ ]:
width = int(root.find("size").find("width").text)
height = int(root.find("size").find("height").text)
print(f"image width: {width}  image height: {height}")

The `findall` method finds all occurrences of a sub-element within a given tag. We can use that method to get all the relevant data for the bounding boxes.

In [ ]:
VimeoVideo("1009519361", h="a5334f4e3c", width=600)

**Task 3.2.4:** Find the labels and coordinates for all the bounding boxes.

In [ ]:
bounding_boxes = []
labels = []
for obj in root.findall("object"):
    label = obj.find("name").text  # REMOVERHS
    labels.append(label)
    bndbox = obj.find("bndbox")
    xmin = int(bndbox.find("xmin").text)  # REMOVERHS
    ymin = int(bndbox.find("ymin").text)  # REMOVERHS
    xmax = int(bndbox.find("xmax").text)  # REMOVERHS
    ymax = int(bndbox.find("ymax").text)  # REMOVERHS
    bounding_boxes.append([xmin, ymin, xmax, ymax])

for label, bounding_box in zip(labels, bounding_boxes):
    print(f"{label}: {bounding_box}")

# Bounding boxes in PyTorch

In [ ]:
VimeoVideo("1009519382", h="c376070230", width=600)

**Task 3.2.5:** Convert bounding boxes to PyTorch tensors.

In [ ]:
# GRADE ME

In [ ]:
bboxes_tensor = torch.tensor(bounding_boxes, dtype=torch.float)  # REMOVERHS
print(bboxes_tensor)

In [ ]:
VimeoVideo("1009519396", h="daaabcf02a", width=600)

**Task 3.2.6:** Create a variable for the image path using `pathlib` syntax.

In [ ]:
image_path = images_dir / "01.jpg"  # REMOVERHS
image = read_image(str(image_path))
print(image.shape)

We can use the `draw_bounding_boxes` function to add the bounding boxes and labels to the image.

In [ ]:
image = draw_bounding_boxes(
    image=image,
    boxes=bboxes_tensor,
    labels=labels,
    width=3,
    fill=False,
    font="arial.ttf",
    font_size=10,
)

In [ ]:
VimeoVideo("1009519402", h="5d2018fefa", width=600)

**Task 3.2.7:** Display the composite image with bounding boxes and labels.

In [ ]:
# INSERT to_pil_image(...)
to_pil_image(image)  # REMOVELINE

# YouTube Video Data

Next, we load YouTube video of traffic in Dhaka, Bangladesh.

In [ ]:
VimeoVideo("1009519406", h="52009fe307", width=600)

**Task 3.2.8:** Create a variable for the video directory using `pathlib` syntax.

In [ ]:
video_dir = Path("data_video")
video_name = "dhaka_traffic.mp4"
video_path = video_dir / video_name  # REMOVERHS

print(video_path)
YouTube  # REMOVELINE # Flake8 ignores the cell below, so use YouTube somewhere

We need to specify the URL of the YouTube video and download the video.

In [ ]:
video_url = "https://www.youtube.com/watch?v=0B2-cR4GEjc&list=WL&index=71&t=22s"
yt = YouTube(video_url)
stream = yt.streams.get_highest_resolution()
stream.download(output_path=video_dir, filename=video_name)

If that does not work, there is a backup copy that can be downloaded.

In [ ]:
if not video_path.exists():
    video_dir.mkdir(exist_ok=True)
    !gcloud storage cp --no-clobber \
        gs://wqu-cv-course-datasets/dhaka_traffic.mp4 \
        $video_dir 

Let's look at the video.

In [ ]:
Video(video_path, embed=True)

We are going to capture still images from the video to make it easier to draw bounding boxes.

In [ ]:
video_capture = cv2.VideoCapture(video_path)

if not video_capture.isOpened():
    print("Error: Could not open video.")
else:
    frame_rate = video_capture.get(cv2.CAP_PROP_FPS)
    frame_count = int(video_capture.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Frame rate: {frame_rate}")
    print(f"Total number of frames: {frame_count:,}")

Let's look the first frame.

In [ ]:
success, first_frame = video_capture.read()
if success:
    plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
    plt.title("First Frame")
    plt.axis("off")
    plt.show()
else:
    print("Error: Could not read frame.")

We were successful in displaying the first frame, however that frame is black. Let's look at a later frame that is more interesting.

In [ ]:
video_capture.set(cv2.CAP_PROP_POS_FRAMES, 100)
success, later_frame = video_capture.read()
if success:
    plt.imshow(cv2.cvtColor(later_frame, cv2.COLOR_BGR2RGB))
    plt.title("First Frame")
    plt.axis("off")
    plt.show()
else:
    print("Error: Could not read frame.")

In [ ]:
VimeoVideo("1009519420", h="3cbbc3fb4c", width=600)

**Task 3.2.9:** Create a directory for the frames using the `pathlib` syntax.

In [ ]:
frames_dir = video_dir / "extracted_frames"  # REMOVERHS
# INSERT frames_dir.mkdir(...)
frames_dir.mkdir(exist_ok=True)  # REMOVELINE

Now we walk through the video, saving selected frames as we go.

In [ ]:
frame_count = 0

while True:
    success, frame = video_capture.read()
    if not success:
        break

    # Save frames at the frame_rate
    if frame_count % frame_rate == 0:
        frame_path = frames_dir / f"frame_{frame_count}.jpg"
        cv2.imwrite(frame_path, frame)

    frame_count += 1

video_capture.release()

We can look at the frames we have extracted and saved using the `display_sample_images` function.

In [ ]:
def display_sample_images(dir_path, sample=5):
    image_list = []
    images = sorted(dir_path.iterdir())
    if images:
        sample_images = images[:sample]
        for sample_image in sample_images:
            image = read_image(str(sample_image))
            resize_transform = transforms.Resize((240, 240))
            image = resize_transform(image)
            image_list.append(image)
    grid = make_grid(image_list, nrow=5)
    image = to_pil_image(grid)
    return image


display_sample_images(frames_dir, sample=10)

# Conclusion

Great work on completing this lesson! We've covered several important aspects of working with image and video data for object detection, focusing on a traffic dataset. Here are the key things we learned:

- Data organization: We explored how to load and organize our project dataset, separating images from their corresponding XML annotations.

- Handling different data sources: We worked with both a pre-existing image dataset and extracted frames from a YouTube video, showing how to prepare data from various sources.

- Video processing: We learned how to extract frames from a video at regular intervals, which is crucial for preparing video data for object detection tasks.

- XML parsing: We demonstrated how to parse XML annotations, which contain valuable information about object locations and classifications in our images.

- Bounding box visualization: We visualized bounding boxes on image data, an essential skill for understanding and validating our object detection results.

In the next lesson, we'll build upon these skills to use a pre-trained model for object detection.

---
This file &#169; 2024 by [WorldQuant University](https://www.wqu.edu/) is licensed under [CC BY-NC-ND 4.0](https://creativecommons.org/licenses/by-nc-nd/4.0/).